# SmolVLA Stage 3: 2-axis fine-tuning and evaluation

This notebook template runs the Stage 3 workflow in Google Colab. It compares the Stage 2 checkpoint, expert-only continuation, and VLM text-path adaptation on a new validation split, then evaluates the validation-selected model once on an independent final split.

The actual 2026-10-17 run, metrics, plots, and checkpoints are in the [Stage 3 Drive folder](https://drive.google.com/drive/folders/1MNAuBtLtAqfq1Kow4ezWldFzRL5VCAXe). The executed notebook is [here](https://colab.research.google.com/drive/1s8RZAZ6JnSXk8noccYtSCVgc1pTbkPWg).

Use a fresh Colab GPU runtime. Stage 2 checkpoint is copied locally and never modified. Final-test data must not be used for model selection.

In [ ]:
import sys, subprocess, importlib.metadata, torch
assert torch.cuda.is_available(), "Select a Colab GPU runtime"
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "lerobot[smolvla,training]==0.6.1", "pillow", "matplotlib", "av"])
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path
import shutil, json
REPO = Path("/content/physical-ai-pointer-demo")
if not REPO.exists():
    subprocess.check_call(["git", "clone", "--depth=1", "--branch",
        "test/vla-two-axis-preflight",
        "https://github.com/ryotamatsuki/physical-ai-pointer-demo.git", str(REPO)])
sys.path.insert(0, str(REPO / "demos" / "vla_pan_tilt"))
from stage3_train_compare import run_data_generation, run_small_overfit, run_expert_continue, run_vlm_adaptation
from stage3_helpers import inspect_dataset, write_csv, build_dataset
from stage3_evaluate import (evaluate_checkpoint, evaluate_paraphrases, evaluate_input_ablation,
    evaluate_instruction_switch, evaluate_swapped_positions, evaluate_state_effect)
from stage3_closed_loop import evaluate_closed_loop
from lerobot.datasets.lerobot_dataset import LeRobotDataset

STAGE2_DRIVE = Path("/content/drive/MyDrive/SmolVLA_2axis_Stage2_20261017/checkpoint_5000")
STAGE2_LOCAL = Path("/content/stage3/input_checkpoint_5000")
STAGE3_DRIVE = Path("/content/drive/MyDrive/SmolVLA_2axis_Stage3_template_run")
if STAGE3_DRIVE.exists():
    raise FileExistsError(f"Choose a fresh Stage 3 Drive folder: {STAGE3_DRIVE}")
for folder in ("checkpoints", "logs", "results", "notebooks"):
    (STAGE3_DRIVE / folder).mkdir(parents=True, exist_ok=True)
assert STAGE2_DRIVE.is_dir(), STAGE2_DRIVE
if not STAGE2_LOCAL.exists():
    shutil.copytree(STAGE2_DRIVE, STAGE2_LOCAL)
assert (STAGE2_LOCAL / "model.safetensors").is_file()
print("Stage 2 source is protected; local copy ready.")

In [ ]:
splits = run_data_generation()
train_root, train_manifest_path, train_manifest = splits["train"]
val_root, val_manifest_path, val_manifest = splits["validation"]
final_root, final_manifest_path, final_manifest = splits["final_test"]
train_hashes = {s["image_sha256"] for s in train_manifest["scenes"]}
val_hashes = {s["image_sha256"] for s in val_manifest["scenes"]}
final_hashes = {s["image_sha256"] for s in final_manifest["scenes"]}
assert train_hashes.isdisjoint(val_hashes)
assert train_hashes.isdisjoint(final_hashes)
assert val_hashes.isdisjoint(final_hashes)
train_ds = LeRobotDataset(repo_id="local/stage3_train", root=train_root)
val_ds = LeRobotDataset(repo_id="local/stage3_validation", root=val_root)
final_ds = LeRobotDataset(repo_id="local/stage3_final_test", root=final_root)
assert len(train_ds) == 20000 and len(val_ds) == 3000 and len(final_ds) == 6000
for split, path in [("train", train_manifest_path), ("validation", val_manifest_path), ("final_test", final_manifest_path)]:
    dest = STAGE3_DRIVE / "results" / f"{split}_manifest.json"
    shutil.copy2(path, dest)
print("Frames:", len(train_ds), len(val_ds), len(final_ds))
print("Scenes:", len(train_manifest["scenes"]), len(val_manifest["scenes"]), len(final_manifest["scenes"]))
print("FPS:", train_ds.meta.info.fps, "State/action:", train_ds.meta.features["observation.state"]["shape"], train_ds.meta.features["action"]["shape"])

## A. Small-set overfit check

Run this first. Treat it as a training-pipeline check, not a generalization result.

In [ ]:
overfit_result = run_small_overfit(STAGE2_LOCAL, steps=500, learning_rate=1e-4, batch_size=2)
print(json.dumps({k:v for k,v in overfit_result.items() if k != "trainable_names"}, indent=2))
shutil.copytree(overfit_result["checkpoint"], STAGE3_DRIVE / "checkpoints" / "overfit_500", dirs_exist_ok=True)
shutil.copy2(overfit_result["log_path"], STAGE3_DRIVE / "logs" / "overfit_500_training.log")

## B and C. Continue from the same Stage 2 weights

These are separate weights-only initializations with fresh optimizer state. B updates the action expert/policy. C includes the VLM text path; the vision encoder stays frozen.

In [ ]:
result_b = run_expert_continue(STAGE2_LOCAL, train_root, steps=2000, learning_rate=1e-5, batch_size=1)
result_c = run_vlm_adaptation(STAGE2_LOCAL, train_root, steps=2000, learning_rate=5e-6, batch_size=1)
for name, result in [("expert_B_2000", result_b), ("vlm_C_2000", result_c)]:
    dest = STAGE3_DRIVE / "checkpoints" / name
    shutil.copytree(result["checkpoint"], dest, dirs_exist_ok=True)
    shutil.copy2(result["log_path"], STAGE3_DRIVE / "logs" / f"{name}_training.log")
    (STAGE3_DRIVE / "results" / f"{name}_training.json").write_text(
        json.dumps({k:v for k,v in result.items() if k != "trainable_names"}, indent=2), encoding="utf-8")
print("B:", result_b["seconds"], "seconds;", result_b["trainable_parameters"], "trainable")
print("C:", result_c["seconds"], "seconds;", result_c["trainable_parameters"], "trainable")

## Select using validation only

After checking the small-set fit and parameter audit, compare the three policies on validation. Freeze the selection before the one-time final test.

In [ ]:
model_paths = {
    "stage2": STAGE2_LOCAL,
    "expert_B": STAGE3_DRIVE / "checkpoints" / "expert_B_2000",
    "vlm_C": STAGE3_DRIVE / "checkpoints" / "vlm_C_2000",
}
validation = {}
for name, checkpoint in model_paths.items():
    rows, summary = evaluate_checkpoint(checkpoint, val_ds, val_manifest, train_ds.meta.stats, label=name)
    validation[name] = summary
    write_csv(STAGE3_DRIVE / "results" / f"{name}_validation.csv", rows)
    (STAGE3_DRIVE / "results" / f"{name}_validation.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(validation, indent=2))
selected_name = max(validation, key=lambda n: validation[n]["success_rate"])
selected_checkpoint = model_paths[selected_name]
print("Selected by validation only:", selected_name, selected_checkpoint)

## Counterfactuals and ideal-actuator closed-loop probe

These probes use validation scenes. The simulated image remains fixed and actuators are ideal; they do not establish physical safety.

In [ ]:
probe_rows = {}
probe_rows["language_switch"] = evaluate_instruction_switch(selected_checkpoint, val_ds, val_manifest, train_ds.meta.stats)
probe_rows["position_swap"] = evaluate_swapped_positions(selected_checkpoint, val_ds, val_manifest, train_ds.meta.stats)
probe_rows["start_state"] = evaluate_state_effect(selected_checkpoint, val_ds, val_manifest, train_ds.meta.stats)
probe_rows["unseen_paraphrases"] = evaluate_paraphrases(selected_checkpoint, val_ds, val_manifest, train_ds.meta.stats)
probe_rows["input_ablation"] = evaluate_input_ablation(selected_checkpoint, val_ds, val_manifest, train_ds.meta.stats)
closed_rows, trajectories = evaluate_closed_loop(selected_checkpoint, val_ds, val_manifest, train_ds.meta.stats, scene_limit=10)
write_csv(STAGE3_DRIVE / "results" / "closed_loop.csv", closed_rows)
(STAGE3_DRIVE / "results" / "counterfactual_summaries.json").write_text(json.dumps(probe_rows, indent=2), encoding="utf-8")
print(json.dumps(probe_rows, indent=2))

## One-time final evaluation

Run only after model selection and diagnostic review. Do not tune against this result. If you change training after seeing this score, reserve a new independent final split.

In [ ]:
final_rows, final_summary = evaluate_checkpoint(
    selected_checkpoint, final_ds, final_manifest, train_ds.meta.stats, label=f"{selected_name}_final")
write_csv(STAGE3_DRIVE / "results" / "final_test.csv", final_rows)
(STAGE3_DRIVE / "results" / "final_test.json").write_text(json.dumps(final_summary, indent=2), encoding="utf-8")
print(json.dumps(final_summary, indent=2))
print("80% success target met:", final_summary["success_rate"] >= 0.80)